# we will create ingestion here i.e. in this notebook

#create a github connection
#import data from github
# create unity catalog, schema, volume first and then
#and land/save to raw volume as it is
github_conn = 
path = '/raw_volume'

data = github_conn.import(github link data)

data.save(path)
print('data saved sucessfully at path')





In [0]:

%sh
rm -rf /tmp/my_repo
git clone https://github.com/nileshpatil1206070/time_n_work_utilization_at_7trents.git /tmp/my_repo

# creating unity catalog bronze schema and raw volume landing

In [0]:
%sql
-- 1. Create Catalog
CREATE CATALOG IF NOT EXISTS 7trents_data_platform;

-- 2. Create Bronze Schema
CREATE SCHEMA IF NOT EXISTS 7trents_data_platform.bronze;

-- 3. Create Volume for raw landing
CREATE VOLUME IF NOT EXISTS 7trents_data_platform.bronze.raw_data;



In [0]:
import shutil
source = "/tmp/my_repo/data_ingest_github"
destination = "/Volumes/7trents_data_platform/bronze/raw_data"

shutil.copytree(source,destination, dirs_exist_ok= True)

print(f'folder data_ingest_github is copied to {destination}')

#above code works perfectly fine--but if data is continously coming and suppose we need to add only latest incremental data  then we need to do some changes as below --dont use code below---just change the cell type from text to python by removing '%md' at start of the cell

============================================================

=============================================================

%sh
# If the repo does not exist locally yet, clone it once.
# If it already exists, just fetch the new files/commits (takes < 1 second).
if [ ! -d "/tmp/my_repo/.git" ]; then
  git clone https://github.com/nileshpatil1206070/time_n_work_utilization_at_7trents.git /tmp/my_repo
else
  cd /tmp/my_repo
  git pull origin main
fi

import os
import shutil

source_base = "/tmp/my_repo/data_ingest_github"
destination_base = "/Volumes/7trents_data_platform/bronze/raw_data"

# 1. Get existing folders in destination volume (if any)
if os.path.exists(destination_base):
    existing_dest_folders = os.listdir(destination_base)
else:
    existing_dest_folders = []

# 2. Get available folders from GitHub source
source_folders = os.listdir(source_base)

# 3. Copy ONLY the folders that haven't been loaded yet
new_folders_count = 0

for folder in source_folders:
    src_folder_path = os.path.join(source_base, folder)
    dest_folder_path = os.path.join(destination_base, folder)
    
    # Check if it is a directory and not already copied
    if os.path.isdir(src_folder_path) and (folder not in existing_dest_folders):
        print(f"New folder detected: {folder}. Ingesting...")
        shutil.copytree(src_folder_path, dest_folder_path)
        new_folders_count += 1
    else:
        print(f"Skipping already ingested folder: {folder}")

if new_folders_count == 0:
    print("No new data to ingest. Everything is up to date!")
else:
    print(f"Successfully ingested {new_folders_count} new date folder(s).")